# 08 — Human performance ceiling (Stage G)

How well does one annotator agree with the other four? Each val image was labelled by five people;
the annotations hold only the number of yes-votes, so a held-out annotator is simulated
(`src/models/ceiling.py`): one of the five is drawn at random per image and output, their yes/no is
the human prediction, and the reference is positive when at least 2 of the other 4 said yes. The
three arms are scored against the same references in every repeat, so humans and models are
compared on identical targets. 200 repeats, seed 42. Regenerates
`reports/results_human_ceiling.md` and `reports/results_human_ceiling.csv`.

**Partitions:** val only (annotation counts and saved val scores). The test partition is never read.

**How to run:** from the `notebooks/` folder, with `DATA_DIR` pointing at the data directory
(defaults to `../data`). `DATA_DIR` must contain `annotations/{train,val}.json` and
`predictions/{classical_gb_224,cnn_finetune,hybrid_finetune}_val.csv`. No GPU needed; under a
minute.

In [2]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
sys.path.insert(0, str(REPO_DIR))
DATA_DIR = Path(os.environ.get("DATA_DIR", REPO_DIR / "data"))
REPORT_PATH = REPO_DIR / "reports" / "results_human_ceiling.md"
RESULTS_CSV = REPO_DIR / "reports" / "results_human_ceiling.csv"
PREDICTIONS_DIR = DATA_DIR / "predictions"

from src.data.splits import MODELLED_FLAWS, get_splits
from src.eda import labels as lb
from src.eda import report as rp
from src.models import ceiling as ce
from src.models import classical as cm

N_REPEATS = 200
CLASSICAL, CNN, HYBRID = "Classical GB @ 224", "CNN fine-tuned", "Hybrid"
MODELS = {CLASSICAL: "classical_gb_224_val.csv", CNN: "cnn_finetune_val.csv", HYBRID: "hybrid_finetune_val.csv"}
ENVIRONMENT = rp.environment_lines(
    gpu_note="not needed: vote counts and saved val scores only",
    seed_note=f"held-out-annotator simulation, {N_REPEATS} repeats")
pd.set_option("display.precision", 3)

print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)
print("\n".join(ENVIRONMENT))

REPO_DIR: /content/drive/MyDrive/Masters/Sem 3/capstone/repo
DATA_DIR: /content/drive/MyDrive/Masters/Sem 3/capstone/data
- Python 3.13.18 on Linux-6.6.159+-x86_64-with-glibc2.39
- RAM: 12.7 GB
- GPU: none (not needed: vote counts and saved val scores only)
- Libraries: numpy 2.4.6, pandas 2.2.3, scikit-learn 1.6.1, torch 2.14.1+cpu, torchvision 0.29.1+cpu
- Seed: 42 (held-out-annotator simulation, 200 repeats)


## 1. Data and annotator agreement
Vote counts (0-5 yes-votes per image and output) for the val partition, the three arm entries'
val scores, and Fleiss' kappa per output (the Stage B agreement measure, here on val).

In [3]:
splits = get_splits(DATA_DIR)
counts = lb.vote_matrix(splits["val"], MODELLED_FLAWS)[cm.OUTPUTS]
scores = {}
for name, filename in MODELS.items():
    s = pd.read_csv(PREDICTIONS_DIR / filename, index_col=0)  # first column: image name
    assert sorted(s.index) == sorted(counts.index), f"{filename}: image set differs from val"
    assert list(s.columns) == cm.OUTPUTS, f"{filename}: columns {list(s.columns)}"
    scores[name] = s.loc[counts.index]
    print(f"{name:<20} {filename:<26} {len(s):,} images")

agreement = lb.agreement_table(splits["val"], MODELLED_FLAWS)
kappa = agreement.iloc[:, 0].rename("Fleiss' kappa")
display(agreement)
sections = [("Data and annotator agreement", (
    f"Val {len(counts):,} images, five annotators each. Agreement per output (Fleiss' kappa, share of "
    "images where all five agree, share at the 2/3 boundary):\n\n" + rp.df_to_markdown(agreement.round(3))))]

Classical GB @ 224   classical_gb_224_val.csv   3,875 images
CNN fine-tuned       cnn_finetune_val.csv       3,875 images
Hybrid               hybrid_finetune_val.csv    3,875 images


,fleiss_kappa,unanimous %,borderline 2-3 %,positive >=2 %
BLR,0.416,48.129,23.252,42.219
BRT,0.218,81.316,4.748,5.781
DRK,0.249,80.903,4.800,6.297
FRM,0.229,28.026,37.703,55.432
OBS,0.194,85.884,2.942,3.535
ROT,0.385,72.387,12.465,17.135
UNREC,0.397,70.735,9.703,15.639


## 2. Simulation
200 repeats of the held-out-annotator draw. Each repeat scores the human and the three models
against that repeat's 4-annotator references.

In [4]:
start = time.time()
runs = ce.human_ceiling(counts, scores, n_repeats=N_REPEATS)
print(f"{N_REPEATS} repeats in {time.time() - start:.0f}s, {len(runs):,} rows")
display(runs.groupby("output", sort=False)["reference base rate"].mean().to_frame("reference base rate (mean)"))

200 repeats in 30s, 5,600 rows


,reference base rate (mean)
output,
BLR,0.371
BRT,0.044
DRK,0.049
FRM,0.477
OBS,0.027
ROT,0.148
UNREC,0.132


## 3. The ceiling: one annotator's F1 against the other four
Human F1 is the ceiling. Models: F1 at the project's fixed 0.5 threshold, and F1 at their best
threshold for these references (**optimistic**: the threshold is chosen on the labels it is scored
against). The fair comparison lies between the two model columns. The 95% range is over repeats.

In [5]:
f1_mean, f1_low, f1_high = ce.summarise(runs, "F1")
best_mean, _, _ = ce.summarise(runs, "best F1")
ceiling = pd.DataFrame({"Fleiss' kappa": kappa.reindex(cm.OUTPUTS),
                        "human F1": f1_mean["human"],
                        "human F1 range": [f"{a:.3f}-{b:.3f}" for a, b in zip(f1_low["human"], f1_high["human"])]})
for name in MODELS:
    ceiling[f"{name} F1 @ 0.5"] = f1_mean[name]
for name in MODELS:
    ceiling[f"{name} best F1"] = best_mean[name]
ceiling[f"{CNN} F1 @ 0.5 - human"] = ceiling[f"{CNN} F1 @ 0.5"] - ceiling["human F1"]
ceiling[f"{CNN} best F1 - human"] = ceiling[f"{CNN} best F1"] - ceiling["human F1"]
display(ceiling)
sections.append(("The ceiling: one annotator against the other four (val)", (
    f"Mean over {N_REPEATS} held-out-annotator repeats; reference = at least 2 of the other 4 annotators. "
    "Model best F1 is optimistic (threshold chosen on the same references).\n\n"
    + rp.df_to_markdown(ceiling.round(3)))))

,Fleiss' kappa,human F1,human F1 range,Classical GB @ 224 F1 @ 0.5,CNN fine-tuned F1 @ 0.5,Hybrid F1 @ 0.5,Classical GB @ 224 best F1,CNN fine-tuned best F1,Hybrid best F1,CNN fine-tuned F1 @ 0.5 - human,CNN fine-tuned best F1 - human
BLR,0.416,0.671,0.657-0.684,0.705,0.773,0.770,0.708,0.776,0.774,0.102,0.105
BRT,0.218,0.327,0.286-0.362,0.298,0.342,0.337,0.454,0.518,0.539,0.015,0.191
DRK,0.249,0.375,0.340-0.409,0.313,0.346,0.344,0.549,0.569,0.581,-0.029,0.194
FRM,0.229,0.607,0.594-0.618,0.620,0.721,0.723,0.674,0.736,0.738,0.113,0.128
OBS,0.194,0.321,0.267-0.367,0.119,0.246,0.252,0.205,0.553,0.548,-0.075,0.232
ROT,0.385,0.577,0.548-0.601,0.298,0.585,0.561,0.302,0.621,0.627,0.008,0.044
UNREC,0.397,0.560,0.538-0.584,0.574,0.597,0.608,0.661,0.713,0.706,0.037,0.153


## 4. How the human and the CNN err
Precision and recall of the held-out annotator against the CNN at 0.5. A single annotator has a
natural operating point; the class-weighted CNN at 0.5 favours recall on the rare flaws.

In [6]:
pr = {}
for stat in ("precision", "recall"):
    mean, _, _ = ce.summarise(runs, stat)
    for who in ("human", CNN):
        pr[f"{who} {stat}"] = mean[who]
pr = pd.DataFrame(pr)[[f"human precision", f"human recall", f"{CNN} precision", f"{CNN} recall"]]
display(pr)
sections.append(("Precision and recall: human vs CNN at 0.5 (val)", rp.df_to_markdown(pr.round(3))))

,human precision,human recall,CNN fine-tuned precision,CNN fine-tuned recall
output,,,,
BLR,0.725,0.625,0.757,0.791
BRT,0.289,0.378,0.216,0.815
DRK,0.339,0.421,0.216,0.862
FRM,0.685,0.546,0.686,0.759
OBS,0.270,0.396,0.144,0.820
ROT,0.626,0.535,0.459,0.808
UNREC,0.565,0.556,0.457,0.861


## 5. Model AP against the 4-annotator references
For continuity with the rest of the project: AP of each arm against the simulated references
(threshold-free). A single annotator's yes/no has no ranking, so it has no comparable AP.

In [7]:
ap_mean, _, _ = ce.summarise(runs[runs["who"] != "human"], "AP")
display(ap_mean)
sections.append(("Model AP against the 4-annotator references (val)", rp.df_to_markdown(ap_mean.round(3))))

who,Classical GB @ 224,CNN fine-tuned,Hybrid
output,,,
BLR,0.757,0.849,0.851
BRT,0.434,0.494,0.517
DRK,0.543,0.604,0.606
FRM,0.615,0.743,0.751
OBS,0.104,0.555,0.538
ROT,0.214,0.637,0.633
UNREC,0.721,0.791,0.789


## 6. Write the report
The findings block in `reports/results_human_ceiling.md` is hand-written after reading the results
and is preserved on every rerun.

In [8]:
summary = runs.groupby(["output", "who"], sort=False)[["reference base rate", "precision", "recall", "F1", "best F1", "AP"]].mean()
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
summary.reset_index().to_csv(RESULTS_CSV, index=False, float_format="%.4f")
out = rp.write_report(REPORT_PATH, sections, title="Stage G: Human performance ceiling",
                      environment=ENVIRONMENT, findings_heading="Findings")
print("Report written to", out)
print("Results table written to", RESULTS_CSV)

Report written to /content/drive/MyDrive/Masters/Sem 3/capstone/repo/reports/results_human_ceiling.md
Results table written to /content/drive/MyDrive/Masters/Sem 3/capstone/repo/reports/results_human_ceiling.csv
